In [ ]:
!pip install -q transformers datasets torch sentencepiece

In [ ]:
from datasets import load_dataset

# Load English-Hindi dataset from Samanantar
dataset = load_dataset("ai4bharat/samanantar", "hi")

# Remove null rows and take a 5,000 sample for fast training
dataset = dataset.filter(lambda x: x['src'] is not None and x['tgt'] is not None)
train_data = dataset['train'].select(range(5000))

README.md:   0%|          | 0.00/11.4k [00:00<?, ?B/s]

hi/train-00000-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00000-of-00008.parquet: downloading bytes:           |  0.00B            

hi/train-00001-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00001-of-00008.parquet: downloading bytes:           |  0.00B            

...

hi/train-00006-of-00008.parquet: downloading bytes:           |  0.00B            

hi/train-00007-of-00008.parquet: reconstructing file:   0%|          |  0.00B /  240MB            

hi/train-00007-of-00008.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10125706 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10125706 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    inputs = examples['src']
    targets = examples['tgt']
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    # Mask padding tokens in targets so loss function ignores them
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map preprocessing across dataset
tokenized_datasets = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    fp16=torch.cuda.is_available(),  # Automatically enables FP16 if GPU is active
    # ... other args
)

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# 1. Load Model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 2. Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 3. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=500,
    save_strategy="no",
    learning_rate=2e-5,
    fp16=False,  # Set to True if using GPU in Colab
    report_to="none",
)

# 4. Trainer (Fixed processing_class argument)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,  # Replaced tokenizer=tokenizer
    data_collator=data_collator,
)

# 5. Start Training
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  306MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,4.052601
1000,3.907802
1500,3.786205
2000,3.839404
2500,3.885672


TrainOutput(global_step=2500, training_loss=3.89433671875, metrics={'train_runtime': 168.4373, 'train_samples_per_second': 29.685, 'train_steps_per_second': 14.842, 'total_flos': 169491824640000.0, 'train_loss': 3.89433671875, 'epoch': 1.0})

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

Step,Training Loss
500,2.899265
1000,3.078652
1500,3.191186
2000,3.460111
2500,3.758375


TrainOutput(global_step=2500, training_loss=3.27751767578125, metrics={'train_runtime': 168.7345, 'train_samples_per_second': 29.632, 'train_steps_per_second': 14.816, 'total_flos': 169491824640000.0, 'train_loss': 3.27751767578125, 'epoch': 1.0})

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try translating a sentence
print(translate("How are you?"))

तुम कैसे हो?


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# 1. Setup Model & Tokenizer
model_checkpoint = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 2. Load & Prepare Dataset (Using standard IITB English-Hindi dataset subset)
raw_dataset = load_dataset("cfilt/iitb-english-hindi", split="train[:5000]")

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["hi"] for ex in examples["translation"]]

    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_dataset.map(preprocess_function, batched=True)

# 3. Training Setup
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=5e-5,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),  # Automatically turns on GPU acceleration if available
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

# 4. Train
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

...

Step,Training Loss
500,0.307626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.20853816104519227, metrics={'train_runtime': 99.4391, 'train_samples_per_second': 150.846, 'train_steps_per_second': 9.443, 'total_flos': 62402652536832.0, 'train_loss': 0.20853816104519227, 'epoch': 3.0})

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test sentences
print(translate("How are you?"))
print(translate("What is your name?"))
print(translate("I am learning machine learning."))

आप कैसे हैं?
आपका GroupWise आईडी क्या है?
मैं मशीन सीख रहा हूँ.


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the fresh base model directly without fine-tuning
model_name = "Helsinki-NLP/opus-mt-en-hi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = base_model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test general sentences
print(translate("How are you?"))
print(translate("What is your name?"))
print(translate("I am learning machine learning."))

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


आप कैसे हैं?


[transformers] Both `max_new_tokens` (=100) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


आपका Windows Live कूटशब्द क्या है?
मैं मशीन सीखने के लिए सीख रहा हूँ.


In [ ]:
trainer.train()

Step,Training Loss
500,0.081921


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.06784911963124625, metrics={'train_runtime': 115.3221, 'train_samples_per_second': 130.07, 'train_steps_per_second': 8.142, 'total_flos': 62402652536832.0, 'train_loss': 0.06784911963124625, 'epoch': 3.0})

In [ ]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Try translating a sentence
print(translate("Welcome to our application."))

हमारे अनुप्रयोग में आपका स्वागत है.


In [ ]:
trainer.train()

Step,Training Loss
500,0.059458


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=939, training_loss=0.0509188208209313, metrics={'train_runtime': 129.386, 'train_samples_per_second': 115.932, 'train_steps_per_second': 7.257, 'total_flos': 62402652536832.0, 'train_loss': 0.0509188208209313, 'epoch': 3.0})

In [ ]:
# Save model and tokenizer to a directory
model.save_pretrained("./my_translation_model")
tokenizer.save_pretrained("./my_translation_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_translation_model/tokenizer_config.json',
 './my_translation_model/vocab.json',
 './my_translation_model/source.spm',
 './my_translation_model/target.spm',
 './my_translation_model/added_tokens.json')

In [ ]:
import os

print(os.listdir())

['.config', 'results', 'my_translation_model', 'translation_model', 'sample_data']


In [ ]:
!zip -r my_translation_model.zip my_translation_model

  adding: my_translation_model/ (stored 0%)
  adding: my_translation_model/source.spm (deflated 51%)
  adding: my_translation_model/config.json (deflated 64%)
  adding: my_translation_model/tokenizer_config.json (deflated 67%)
  adding: my_translation_model/target.spm (deflated 60%)
  adding: my_translation_model/generation_config.json (deflated 43%)
  adding: my_translation_model/model.safetensors (deflated 7%)
  adding: my_translation_model/vocab.json (deflated 76%)


In [ ]:
import os

print(os.path.exists("my_translation_model.zip"))

True


In [ ]:
from google.colab import files

files.download("my_translation_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        print(os.path.join(root, file))

/content/.config/gce
/content/.config/default_configs.db
/content/.config/active_config
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/.last_survey_prompt.yaml
/content/.config/.last_update_check.json
/content/.config/config_sentinel
/content/.config/configurations/config_default
/content/.config/logs/2026.08.20/13.35.23.509509.log
/content/.config/logs/2026.08.20/13.35.45.291038.log
/content/.config/logs/2026.08.20/13.34.59.217877.log
/content/.config/logs/2026.08.20/13.35.46.039993.log
/content/.config/logs/2026.08.20/13.35.32.358199.log
/content/.config/logs/2026.08.20/13.35.34.689503.log
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/mnist_train_small.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv
